# LightGBM v2 — Kaskadenfeature

**Erweiterung des LightGBM v1-Modells** um den Kaskadenindikator `prev_trip_delay`.

**Motivation aus der Analyse:**
Der räumliche Analyseteil (`03_analysis_4-spatial.ipynb`) hat gezeigt, dass der Pearson-Korrelationskoeffizient zwischen dem Delay an Halt N und Halt N+1 netzweit r ≥ 0.85 beträgt. Verspätung kaskadiert fast vollständig von Halt zu Halt — und v1 hat dieses Signal noch nicht genutzt.

**Neue Features:**
| Feature | Beschreibung | Begründung |
|:---|:---|:---|
| `prev_trip_delay` | Arrival Delay am vorherigen Halt desselben Trips | Direkter Kaskadenindikator (r ≥ 0.85) |
| `stop_sequence_pct` | Position entlang der Linie (0 = Anfang · 1 = Ende) | Akkumulationseffekt: Delay wächst zur Peripherie |

**Hinweis:** `prev_trip_delay` verwendet den *tatsächlich gemessenen* Delay des Vorgänger-Halts — das ist kein Leakage für den operativen Anwendungsfall (Echtzeit-Prognose Halt für Halt), aber kein Feature für "Prognose vor Fahrtbeginn".

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import lightgbm as lgb
import numpy as np
import pandas as pd
from pathlib import Path

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("06_prediction_4-model_v2")

processed_dir  = Path(str(TRAIN)).parent
models_dir     = processed_dir.parent / "models"
models_dir.mkdir(exist_ok=True)

train_features_path = str(TRAIN)                              # train_features.parquet
train_final_path    = str(TRAIN).replace("train_features", "train_final")
test_features_path  = str(TEST)                               # test_features.parquet
test_final_path     = str(TEST).replace("test_features", "test_final")

train_v2_path = str(processed_dir / "train_final_v2.parquet")
test_v2_path  = str(processed_dir / "test_final_v2.parquet")

BASELINE_MAE = 50.0   # Stop Mean — aus 06_prediction_1-baseline
V1_VAL_MAE   = 49.05  # LightGBM v1 Validation MAE
V1_TEST_MAE  = 45.7   # LightGBM v1 Test MAE

## Feature Engineering v2

### Strategie

`train_final.parquet` enthält bereits alle engineerten Features — aber keine `stop_sequence` (war als ID-Spalte ausgeschlossen). Wir holen sie aus `train_features.parquet` und joinen sie, um dann `prev_trip_delay` und `stop_sequence_pct` zu berechnen.

**Join-Key:** `(trip_id, operating_date, stop_name)` — eindeutig, da ein Tram denselben Halt pro Trip/Tag nur einmal bedient.

In [ ]:
def add_cascade_features(final_path: str, features_path: str) -> pl.DataFrame:
    """Fügt prev_trip_delay + stop_sequence_pct zu einem final-Parquet hinzu.
    
    Vorgehen:
      1. stop_sequence aus features-Parquet (nur nicht-canceled Zeilen)
      2. Left-Join auf (trip_id, operating_date, stop_name)
      3. Sort by (trip_id, operating_date, stop_sequence)
      4. prev_trip_delay = shift(1).over([trip_id, operating_date]), Null → 0.0
      5. stop_sequence_pct = stop_sequence / n_stops_line
    """
    print(f"Lade {Path(final_path).name} ...")
    df = pl.read_parquet(final_path)
    print(f"  {len(df):,} rows, {len(df.columns)} cols")

    print(f"Lade stop_sequence aus {Path(features_path).name} ...")
    seq_df = (
        pl.read_parquet(
            features_path,
            columns=["trip_id", "operating_date", "stop_name", "stop_sequence", "canceled"],
        )
        .filter(pl.col("canceled") == False)
        .drop("canceled")
        # Deduplizieren — sicherheitshalber (sollte eindeutig sein)
        .unique(subset=["trip_id", "operating_date", "stop_name"], keep="first")
    )
    print(f"  stop_sequence lookup: {len(seq_df):,} rows")

    print("Join + Kaskaden-Berechnung ...")
    result = (
        df
        .join(seq_df, on=["trip_id", "operating_date", "stop_name"], how="left")
        .sort(["trip_id", "operating_date", "stop_sequence"])
        .with_columns(
            # Kaskadenindikator: Delay am vorherigen Halt desselben Trips
            pl.col("arrival_delay")
              .shift(1)
              .over(["trip_id", "operating_date"])
              .fill_null(0.0)   # erster Halt eines Trips → kein Vorgänger → 0
              .alias("prev_trip_delay"),
            # Positions-Feature: 0 = Anfangshalt · 1 = Endhalt
            (pl.col("stop_sequence") / pl.col("n_stops_line"))
              .alias("stop_sequence_pct"),
        )
    )

    null_count = result["stop_sequence"].null_count()
    if null_count > 0:
        print(f"  ⚠️  {null_count:,} Zeilen ohne stop_sequence (Join-Fehler?) — werden behalten")
    print(f"  ✓  {len(result):,} rows · neue Spalten: prev_trip_delay, stop_sequence_pct, stop_sequence")
    return result


train_v2 = add_cascade_features(train_final_path, train_features_path)
test_v2  = add_cascade_features(test_final_path,  test_features_path)

In [ ]:
# Schnell-Check: Korrelation prev_trip_delay ↔ arrival_delay
r = train_v2.select(
    pl.corr("prev_trip_delay", "arrival_delay")
).item()
print(f"Pearson r(prev_trip_delay, arrival_delay) = {r:.4f}")
print()

# Verteilung des neuen Features
show_df(
    train_v2.select("prev_trip_delay").describe().to_pandas()
)

In [ ]:
from pathlib import Path as _Path

_train_v2_exists = _Path(train_v2_path).exists()
_test_v2_exists  = _Path(test_v2_path).exists()

if _train_v2_exists and _test_v2_exists:
    print("train_final_v2.parquet + test_final_v2.parquet bereits vorhanden — Export übersprungen.")
    print(f"  {train_v2_path}")
    print(f"  {test_v2_path}")
else:
    print("Exportiere train_final_v2.parquet ...")
    train_v2.write_parquet(train_v2_path)
    print(f"  ✓  {train_v2_path}")

    print("Exportiere test_final_v2.parquet ...")
    test_v2.write_parquet(test_v2_path)
    print(f"  ✓  {test_v2_path}")

## Feature Set v2

Vergleich v1 vs. v2 — was ist neu, was bleibt gleich.

In [ ]:
TARGET  = "arrival_delay"

# Gleiche Exclude-Liste wie v1, ergänzt um interne Hilfsspalten
EXCLUDE_V2 = [
    TARGET,
    "departure_delay",
    "delay_delta",
    "canceled",
    "trip_id",
    "operating_date",
    "event_name",
    "stop_lat",
    "stop_lon",
    "stop_sequence",   # roh — durch stop_sequence_pct abgedeckt
]

FEATURES_V2 = [c for c in train_v2.columns if c not in EXCLUDE_V2]
CAT_COLS_V2 = [c for c in ["line_name", "stop_name", "event_type", "season", "gtfs_year"]
               if c in FEATURES_V2]

# v1 Feature-Set (aus Metadaten)
import json as _json
with open(models_dir / "lgbm_v1_meta.json") as f:
    v1_meta = _json.load(f)
FEATURES_V1 = v1_meta["features"]

new_features = [f for f in FEATURES_V2 if f not in FEATURES_V1]
removed      = [f for f in FEATURES_V1 if f not in FEATURES_V2]

print(f"Features v1: {len(FEATURES_V1)}")
print(f"Features v2: {len(FEATURES_V2)}")
print(f"\n  Neu in v2:     {new_features}")
print(f"  Entfernt:      {removed if removed else '—'}")

## Validation Split

Identischer temporaler Split wie v1:
* **Train:**      2023-01 bis 2024-06
* **Validation:** 2024-07 bis 2024-12
* **Test:**       2025 (nie gesehen während Training)

In [ ]:
def to_lgb_df(pl_df: pl.DataFrame, features: list, cat_cols: list) -> pd.DataFrame:
    pdf = pl_df.select(features).to_pandas()
    for col in cat_cols:
        if col in pdf.columns:
            pdf[col] = pdf[col].astype("category")
    return pdf


val_mask = (
    (train_v2["operating_date"].dt.year() == 2024)
    & (train_v2["operating_date"].dt.month() >= 7)
)
train_sub = train_v2.filter(~val_mask)
val_sub   = train_v2.filter(val_mask)

print(f"Train subset:      {len(train_sub):,} rows")
print(f"Validation subset: {len(val_sub):,} rows")

print("Konvertiere zu Pandas ...")
X_train = to_lgb_df(train_sub, FEATURES_V2, CAT_COLS_V2)
y_train = train_sub[TARGET].to_numpy()
X_val   = to_lgb_df(val_sub, FEATURES_V2, CAT_COLS_V2)
y_val   = val_sub[TARGET].to_numpy()
print("Fertig.")

## Training: LightGBM v2

Gleiche Hyperparameter wie v1 — nur das Feature Set ist erweitert. So ist der Vergleich sauber: jede Verbesserung kommt ausschliesslich aus den neuen Features.

In [ ]:
_lgbm_v2_path      = models_dir / "lgbm_v2.txt"
_lgbm_v2_meta_path = models_dir / "lgbm_v2_meta.json"

if _lgbm_v2_path.exists():
    # Modell bereits trainiert — laden statt neu trainieren (~2 s statt ~30 min)
    import json as _json_load
    print(f"Modell gefunden — lade {_lgbm_v2_path.name} ...")
    model_v2 = lgb.Booster(model_file=str(_lgbm_v2_path))
    # params + best_score: aus Meta-JSON laden (beim Booster-Load nicht im Objekt)
    if _lgbm_v2_meta_path.exists():
        with open(_lgbm_v2_meta_path) as _f:
            _m = _json_load.load(_f)
        params = _m.get("params", {})
        try:
            _ = model_v2.best_score["valid_0"]["l1"]
        except (AttributeError, KeyError, TypeError):
            model_v2.best_score = {"valid_0": {"l1": _m["val_mae"]}}
    else:
        params = {}
    print(f"Geladen ✓  best_iteration={model_v2.best_iteration}  "
          f"val_mae={model_v2.best_score['valid_0']['l1']:.2f}s")
else:
    lgb_train = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    lgb_val   = lgb.Dataset(X_val,   label=y_val,   reference=lgb_train, free_raw_data=False)

    params = {
        "objective":        "regression_l1",
        "metric":           "mae",
        "num_leaves":       63,
        "learning_rate":    0.05,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq":     5,
        "min_child_samples": 50,
        "verbose":          -1,
        "n_jobs":           -1,
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
    ]

    print("Training startet ...")
    model_v2 = lgb.train(
        params,
        lgb_train,
        num_boost_round=1000,
        valid_sets=[lgb_val],
        callbacks=callbacks,
    )
    print(f"\nBeste Iteration: {model_v2.best_iteration}")
    print(f"Bestes Val-MAE:  {model_v2.best_score['valid_0']['l1']:.2f}s")

## Ergebnis: v1 vs. v2

In [ ]:
v2_val_mae = model_v2.best_score["valid_0"]["l1"]

print("=" * 42)
print(f"  Baseline (Stop Mean):  {BASELINE_MAE:.1f} s")
print(f"  LightGBM v1 Val MAE:   {V1_VAL_MAE:.1f} s  (Δ {BASELINE_MAE - V1_VAL_MAE:+.1f} s)")
print(f"  LightGBM v2 Val MAE:   {v2_val_mae:.2f} s  (Δ {BASELINE_MAE - v2_val_mae:+.2f} s)")
print(f"  v2 vs. v1:             {V1_VAL_MAE - v2_val_mae:+.2f} s")
print("=" * 42)

if v2_val_mae < V1_VAL_MAE:
    print(f"\n✅  v2 verbessert v1 um {V1_VAL_MAE - v2_val_mae:.2f} s — Kaskadenfeature hilft!")
else:
    print(f"\n⚠️  v2 schlechter als v1 um {v2_val_mae - V1_VAL_MAE:.2f} s — Feature-Analyse nötig")

## Feature Importance (Gain)

Steht `prev_trip_delay` unter den Top-Features? Das würde bestätigen, dass das Modell die Kaskade tatsächlich nutzt.

In [ ]:
import plotly.express as px

importance = pd.DataFrame({
    "feature":    model_v2.feature_name(),
    "importance": model_v2.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(20)

# Kaskadenfeature hervorheben
importance["color"] = importance["feature"].apply(
    lambda f: "Kaskadenfeature (neu)" if f in ["prev_trip_delay", "stop_sequence_pct"]
    else "Bestehendes Feature"
)

fig = px.bar(
    importance,
    x="importance",
    y="feature",
    orientation="h",
    color="color",
    color_discrete_map={
        "Kaskadenfeature (neu)": "#de425b",
        "Bestehendes Feature":   "#4c72b0",
    },
    title="Top 20 Feature Importance (Gain) — LightGBM v2",
    labels={"importance": "Gain", "feature": "", "color": ""},
)
fig.update_layout(
    yaxis=dict(autorange="reversed"),
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

show_df(importance.drop(columns="color").reset_index(drop=True))

## SHAP-Werte

SHAP (SHapley Additive exPlanations) erklärt, welchen Beitrag jedes Feature für eine einzelne Vorhersage leistet — aussagekräftiger als Gain-Importance, die nur die globale Nutzung misst.

Benötigt: `uv pip install -e ".[dsc]"`

In [ ]:
try:
    import shap
    import matplotlib.pyplot as plt

    # SHAP auf einer repräsentativen Stichprobe berechnen (5000 Zeilen)
    SHAP_SAMPLE = 5000
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X_val), size=min(SHAP_SAMPLE, len(X_val)), replace=False)
    X_shap = X_val.iloc[idx].copy()

    print(f"Berechne SHAP auf {len(X_shap):,} Stichproben ...")
    explainer = shap.TreeExplainer(model_v2)
    shap_values = explainer.shap_values(X_shap)

    plt.figure(figsize=(10, 7))
    shap.summary_plot(
        shap_values, X_shap,
        max_display=20,
        show=False,
        plot_type="dot",
    )
    plt.title("SHAP Summary — LightGBM v2 (Stichprobe n=5000)", fontsize=12)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("shap nicht installiert — überspringen.")
    print("Installation: uv pip install -e '.[dsc]'")

## Test-Evaluation

In [ ]:
print("Lade Testdaten ...")
test_pl = pl.read_parquet(test_v2_path)
X_test  = to_lgb_df(test_pl, FEATURES_V2, CAT_COLS_V2)
y_test  = test_pl[TARGET].to_numpy()

test_pred_v2 = model_v2.predict(X_test, num_iteration=model_v2.best_iteration)

test_mae_v2  = np.abs(y_test - test_pred_v2).mean()
test_rmse_v2 = np.sqrt(((y_test - test_pred_v2) ** 2).mean())
test_mbe_v2  = (test_pred_v2 - y_test).mean()          # Mean Bias Error
test_otp_v2  = (np.abs(y_test - test_pred_v2) <= 60).mean()

print()
print("=" * 48)
print(f"  Test MAE:        {test_mae_v2:.1f} s  (v1: {V1_TEST_MAE:.1f} s · Δ {V1_TEST_MAE - test_mae_v2:+.1f} s)")
print(f"  Test RMSE:       {test_rmse_v2:.1f} s")
print(f"  MBE (Bias):      {test_mbe_v2:+.1f} s  (v1: +8.3 s)")
print(f"  OTP (±60 s):     {test_otp_v2:.1%}")
print(f"  Baseline:        {BASELINE_MAE:.1f} s")
print("=" * 48)

if test_mbe_v2 > 5:
    print(f"\n⚠️  MBE {test_mbe_v2:+.1f} s — Modell unterschätzt Verspätung systematisch → Bias-Analyse!")

## Fehleranalyse

MAE nach Tageszeit, Linie und Wetter — identische Aufteilung wie v1 für direkten Vergleich.

In [ ]:
import matplotlib.pyplot as plt

err_df = test_pl.with_columns(
    pl.lit(test_pred_v2.astype("float32")).alias("predicted"),
    pl.lit(y_test.astype("float32")).alias("actual"),
).with_columns(
    (pl.col("predicted") - pl.col("actual")).abs().alias("abs_error")
).to_pandas()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAE nach Stunde
mae_hour = err_df.groupby("hour")["abs_error"].mean().sort_index()
axes[0].bar(mae_hour.index, mae_hour.values, color="#4c72b0", alpha=0.8)
axes[0].axhline(test_mae_v2, color="#de425b", ls="--", lw=1.5, label=f"Ø MAE {test_mae_v2:.1f}s")
axes[0].set_title("MAE nach Tageszeit"); axes[0].set_xlabel("Stunde"); axes[0].legend()

# MAE nach Linie
mae_line = (
    err_df.groupby("line_name")["abs_error"].mean()
    .reset_index()
    .sort_values("abs_error", ascending=False)
)
colors = ["#de425b" if v > test_mae_v2 else "#55a868" for v in mae_line["abs_error"]]
axes[1].bar(mae_line["line_name"].astype(str), mae_line["abs_error"], color=colors, alpha=0.85)
axes[1].axhline(test_mae_v2, color="black", ls="--", lw=1.2)
axes[1].set_title("MAE nach Linie"); axes[1].set_xlabel("Linie")
axes[1].tick_params(axis="x", rotation=45)

# MAE nach Wetter
weather_labels  = ["Normal", "Regen", "Starkregen", "Schnee"]
weather_filters = [
    (~err_df["has_rain"]) & (~err_df["has_snow"]),
    err_df["has_rain"],
    err_df["has_heavy_rain"],
    err_df["has_snow"],
]
mae_weather = [err_df.loc[f, "abs_error"].mean() for f in weather_filters]
axes[2].bar(weather_labels, mae_weather, color=["#4c72b0","#937860","#dd8452","#8172b2"], alpha=0.85)
axes[2].axhline(test_mae_v2, color="black", ls="--", lw=1.2)
axes[2].set_title("MAE nach Wetter")

plt.suptitle("Fehleranalyse LightGBM v2", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Bias-Analyse & Kalibrierung

v1 hatte MBE +8.3 s — das Modell unterschätzte Verspätung systematisch. Hier prüfen wir, ob v2 das verbessert, und wenden falls nötig eine **Isotonic Regression** als Post-hoc-Kalibrierung an.

> **Isotonic Regression:** Monotone, nicht-parametrische Kalibrierung — passt die Predicted-Values so an, dass der Bias auf dem Val-Set verschwindet, ohne die Ranking-Güte zu beeinträchtigen.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Kalibrierung auf Validation-Set trainieren
val_pred_v2 = model_v2.predict(X_val, num_iteration=model_v2.best_iteration)

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(val_pred_v2, y_val)

# Kalibrierte Test-Predictions
test_pred_v2_cal = iso.predict(test_pred_v2)

cal_mae = np.abs(y_test - test_pred_v2_cal).mean()
cal_mbe = (test_pred_v2_cal - y_test).mean()

print("Bias-Kalibrierung (Isotonic Regression)")
print(f"  Vor Kalibrierung: MAE {test_mae_v2:.1f} s · MBE {test_mbe_v2:+.1f} s")
print(f"  Nach Kalibrierung: MAE {cal_mae:.1f} s · MBE {cal_mbe:+.1f} s")
print()
if cal_mae < test_mae_v2:
    print(f"✅  Kalibrierung verbessert MAE um {test_mae_v2 - cal_mae:.1f} s")
else:
    print("ℹ️  Kalibrierung verändert MAE kaum — Bias war kein kritisches Problem")

# Predicted vs Actual (unkalibriert vs. kalibriert)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sample = np.random.default_rng(42).choice(len(y_test), 3000, replace=False)

for ax, preds, title in [
    (axes[0], test_pred_v2,     f"v2 unkalibriert (MBE {test_mbe_v2:+.1f} s)"),
    (axes[1], test_pred_v2_cal, f"v2 kalibriert   (MBE {cal_mbe:+.1f} s)"),
]:
    ax.scatter(y_test[sample], preds[sample], alpha=0.15, s=3, color="#4c72b0")
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, "r--", lw=1.5, label="Ideal")
    ax.set_xlabel("Actual (s)"); ax.set_ylabel("Predicted (s)")
    ax.set_title(title); ax.legend()

plt.suptitle("Predicted vs. Actual — v2 (Stichprobe n=3000)", fontsize=12)
plt.tight_layout()
plt.show()

## Export

In [ ]:
import json as _json
from sklearn.utils import estimator_html_repr
import joblib

# LightGBM v2 Modell
model_v2_path = models_dir / "lgbm_v2.txt"
model_v2.save_model(str(model_v2_path))
print(f"Modell gespeichert: {model_v2_path}")

# Isotonic-Kalibrierer
iso_path = models_dir / "lgbm_v2_calibrator.joblib"
joblib.dump(iso, iso_path)
print(f"Kalibrierer gespeichert: {iso_path}")

# Metadaten
meta_v2 = {
    "model":          "lgbm_v2",
    "best_iteration": model_v2.best_iteration,
    "val_mae":        round(v2_val_mae, 2),
    "test_mae":       round(test_mae_v2, 2),
    "test_mae_cal":   round(cal_mae, 2),
    "test_mbe":       round(test_mbe_v2, 2),
    "test_mbe_cal":   round(cal_mbe, 2),
    "baseline_mae":   BASELINE_MAE,
    "v1_test_mae":    V1_TEST_MAE,
    "params":         params,
    "features":       FEATURES_V2,
    "new_features":   new_features,
    "cat_cols":       CAT_COLS_V2,
    "target":         TARGET,
}
meta_v2_path = models_dir / "lgbm_v2_meta.json"
with open(meta_v2_path, "w", encoding="utf-8") as f:
    _json.dump(meta_v2, f, indent=2, ensure_ascii=False)
print(f"Metadaten gespeichert: {meta_v2_path}")

# Test-Predictions v2 (unkalibriert + kalibriert)
pred_v2_df = pl.DataFrame({
    "actual":         y_test,
    "predicted_v2":   test_pred_v2.astype("float32"),
    "predicted_v2_cal": test_pred_v2_cal.astype("float32"),
    "line_name":      test_pl["line_name"],
    "stop_name":      test_pl["stop_name"],
    "hour":           test_pl["hour"],
    "month":          test_pl["month"],
    "has_rain":       test_pl["has_rain"],
    "has_snow":       test_pl["has_snow"],
    "has_event":      test_pl["has_event"],
})
pred_v2_path = processed_dir / "test_predictions_v2.parquet"
pred_v2_df.write_parquet(pred_v2_path)
print(f"Predictions gespeichert: {pred_v2_path}")

print(f"\n{'='*48}")
print(f"  LightGBM v2 — Zusammenfassung")
print(f"  Test MAE:       {test_mae_v2:.1f} s  (Baseline: {BASELINE_MAE} s · v1: {V1_TEST_MAE} s)")
print(f"  Nach Kalib.:    {cal_mae:.1f} s")
print(f"  MBE vorher:     {test_mbe_v2:+.1f} s")
print(f"  MBE nachher:    {cal_mbe:+.1f} s")
print(f"{'='*48}")